In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga teknik ng error mitigation at suppression

> **Note:** Ang beta release ng bagong execution model ay available na. Ang directed execution model ay nagbibigay ng mas maraming flexibility kapag kina-customize ang iyong error mitigation workflow. Tingnan ang gabay na [Directed execution model](/guides/directed-execution-model) para sa karagdagang impormasyon.


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda namin ang paggamit ng mga bersyong ito o mas bago.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang mga teknik ng error mitigation at error suppression ay ginagamit upang mapabuti ang kalidad ng resulta kapag nagpa-scale up sa mas malalaking workload. Ang pahinang ito ay nagbibigay ng mataas na antas na paliwanag ng mga teknik ng error suppression at error mitigation na available sa pamamagitan ng Qiskit Runtime.

Ang sumusunod na cell ay ini-import ang Estimator primitive at gumagawa ng backend na gagamitin para sa pag-initialize ng Estimator sa mga susunod na code cell.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamical decoupling
Ang mga quantum circuit ay isinasagawa sa IBM&reg; hardware bilang mga sequence ng microwave pulse na kailangang i-schedule at patakbuhin sa tiyak na mga agwat ng oras.
Sa kasamaang palad, ang mga hindi gustong interaksyon sa pagitan ng mga qubit ay maaaring humantong sa mga coherent error sa mga idling qubit. Gumagana ang dynamical decoupling sa pamamagitan ng paglalagay ng mga pulse sequence sa mga idling qubit upang halos kanselahin ang epekto ng mga error na ito. Ang bawat na-insert na pulse sequence ay katumbas ng isang identity operation, ngunit ang pisikal na presensya ng mga pulse ay may epekto ng pag-suppress ng mga error.
Maraming posibleng pagpipilian ng mga pulse sequence, at kung aling sequence ang mas mabuti para sa bawat partikular na kaso ay nananatiling isang [aktibong larangan ng pananaliksik](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Tandaan na ang dynamical decoupling ay pangunahing kapaki-pakinabang para sa mga circuit na naglalaman ng mga puwang kung saan ang ilang qubit ay idle nang walang anumang operasyon na kumikilos sa kanila. Kung ang mga operasyon sa circuit ay napakadense, kung kaya't ang lahat ng qubit ay abala karamihan sa oras, ang pagdaragdag ng mga dynamical decoupling pulse ay maaaring hindi mapabuti ang performance. Sa katunayan, maaari pa nitong mapasama ang performance dahil sa mga kakulangan ng mismong mga pulse.

Ang diagram sa ibaba ay naglalarawan ng dynamical decoupling gamit ang isang XX pulse sequence. Ang abstract circuit sa kaliwa ay nima-map sa isang microwave pulse schedule sa kanang tuktok. Ang kanang ibaba ay naglalarawan ng parehong schedule, ngunit may isang sequence ng dalawang X pulse na ipinasok sa panahon ng idle period ng unang qubit.

![Depiction of dynamical decoupling](../docs/images/guides/error-mitigation-explanation/dd.avif)

Maaaring i-enable ang dynamical decoupling sa pamamagitan ng pag-set ng `enable` sa `True` sa [mga opsyon ng dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Ang opsyon na `sequence_type` ay maaaring gamitin upang pumili mula sa ilang iba't ibang pulse sequence. Ang default na uri ng sequence ay `"XX"`.

Ang sumusunod na code cell ay nagpapakita kung paano i-enable ang dynamical decoupling para sa Estimator at pumili ng isang dynamical decoupling sequence.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Pauli twirling
Ang twirling, na kilala rin bilang [randomized compiling](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), ay isang malawakang ginagamit na teknik para sa pag-convert ng mga arbitrary na noise channel sa mga noise channel na may mas tiyak na istraktura.

Ang Pauli twirling ay isang espesyal na uri ng twirling na gumagamit ng mga Pauli operation. Mayroon itong epekto ng pag-transform ng anumang quantum channel sa isang Pauli channel. Kapag ginawa nang mag-isa, maaari nitong mapagaan ang coherent noise dahil ang coherent noise ay may tendency na mag-accumulate nang quadratically kasabay ng bilang ng mga operasyon, samantalang ang Pauli noise ay nag-a-accumulate nang linearly. Ang Pauli twirling ay madalas na pinagsama sa iba pang mga teknik ng error mitigation na mas gumagana sa Pauli noise kaysa sa arbitrary na noise.

Ang Pauli twirling ay isinasagawa sa pamamagitan ng pag-sandwich ng isang piniling hanay ng mga gate gamit ang mga random na piniling single-qubit Pauli gate sa paraang nananatiling pareho ang ideal na epekto ng gate. Ang resulta ay ang isang circuit ay pinapalitan ng isang random na ensemble ng mga circuit, lahat ay may parehong ideal na epekto. Kapag nagsa-sample ng circuit, ang mga sample ay kinukuha mula sa maraming random na instance, hindi lang sa iisa.

![Depiction of Pauli twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Dahil karamihan sa mga error sa kasalukuyang quantum hardware ay nagmumula sa mga two-qubit gate, ang teknik na ito ay madalas na inilalapat nang eksklusibo sa mga (native) two-qubit gate. Ang sumusunod na diagram ay naglalarawan ng ilang Pauli twirl para sa mga CNOT at ECR gate. Ang bawat circuit sa loob ng isang hilera ay may parehong ideal na epekto.

![Depiction of gate twirls](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Maaaring i-enable ang Pauli twirling sa pamamagitan ng pag-set ng `enable_gates` sa `True` sa [mga opsyon ng twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Kasama sa iba pang kapansin-pansing mga opsyon ang:

- `num_randomizations`: Ang bilang ng mga circuit instance na kukuhanin mula sa ensemble ng mga twirled circuit.
- `shots_per_randomization`: Ang bilang ng shots na isa-sample mula sa bawat circuit instance.

Ang sumusunod na code cell ay nagpapakita kung paano i-enable ang Pauli twirling at i-set ang mga opsyong ito para sa estimator. Wala sa mga opsyong ito ang kinakailangang i-set nang explicitly.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Twirled readout error extinction (TREX)
Ang [Twirled readout error extinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) ay nagpapagaan ng epekto ng mga measurement error para sa pagtatantya ng mga expectation value ng Pauli observable.
Batay ito sa konsepto ng twirled measurements, na naaabot sa pamamagitan ng random na pagpapalit ng mga measurement gate ng isang sequence ng (1) isang Pauli X gate, (2) isang measurement, at (3) classical bit flip. Tulad ng sa standard gate twirling, ang sequence na ito ay katumbas ng isang simpleng measurement kung walang noise, tulad ng inilalarawan sa sumusunod na diagram:

![Depiction of measurement twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

Sa presensya ng readout error, ang measurement twirling ay may epekto ng pag-diagonalize ng readout-error transfer matrix, na ginagawa itong madaling i-invert. Ang pagtatantya ng readout-error transfer matrix ay nangangailangan ng pagpapatupad ng karagdagang mga calibration circuit, na nagdudulot ng maliit na overhead.

Maaaring i-enable ang TREX sa pamamagitan ng pag-set ng `measure_mitigation` sa `True` sa [mga opsyon ng Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para sa Estimator. Ang mga opsyon para sa measurement noise learning ay inilalarawan [dito](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Tulad ng gate twirling, maaari mong i-set ang bilang ng mga circuit randomization at ang bilang ng shots bawat randomization.

Ang sumusunod na code cell ay nagpapakita kung paano i-enable ang TREX at i-set ang mga opsyong ito para sa estimator. Wala sa mga opsyong ito ang kinakailangang i-set nang explicitly.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Zero-noise extrapolation (ZNE)
Ang Zero-noise extrapolation (ZNE) ay isang teknik para mapagaan ang mga error sa pagtatantya ng mga expectation value ng mga observable. Habang madalas nitong pinapabuti ang mga resulta, hindi garantisado na makakagawa ito ng walang bias na resulta.

Ang ZNE ay binubuo ng dalawang yugto:

1. _Noise amplification_: Ang orihinal na quantum circuit ay isinasagawa nang maraming beses sa iba't ibang antas ng noise.
2. _Extrapolation_: Ang ideal na resulta ay tinatantya sa pamamagitan ng pag-extrapolate ng mga maingay na resulta ng expectation value patungo sa zero-noise limit.

Ang parehong noise amplification at extrapolation na yugto ay maaaring ipatupad sa maraming iba't ibang paraan. Ang Qiskit Runtime ay nagpapatupad ng noise amplification sa pamamagitan ng "digital gate folding," na nangangahulugang ang mga two-qubit gate ay pinapalitan ng mga katumbas na sequence ng gate at ang inverse nito. Halimbawa, ang pagpapalit ng isang unitary $U$ ng $U U^\dagger U$ ay magbubunga ng noise amplification factor na 3. Para sa extrapolation, maaari kang pumili mula sa isa sa ilang functional form, kasama ang linear fit o exponential fit.
Ang larawan sa ibaba ay naglalarawan ng digital gate folding sa kaliwa, at ang extrapolation procedure sa kanan.

![Depiction of ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

Maaaring i-enable ang ZNE sa pamamagitan ng pag-set ng `zne_mitigation` sa `True` sa [mga opsyon ng Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para sa Estimator.
Ang mga opsyon ng Qiskit Runtime para sa ZNE ay inilalarawan [dito](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Ang mga sumusunod na opsyon ay kapansin-pansin:

- `noise_factors`: Ang mga noise factor na gagamitin para sa noise amplification.
- `extrapolator`: Ang functional form na gagamitin para sa extrapolation.

Ang sumusunod na code cell ay nagpapakita kung paano i-enable ang ZNE at i-set ang mga opsyong ito para sa estimator. Wala sa mga opsyong ito ang kinakailangang i-set nang explicitly.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Probabilistic Error Amplification (PEA)
Isa sa mga pangunahing hamon sa ZNE ay ang tumpak na pag-amplify ng noise na nakakaapekto sa target na circuit. Ang gate folding ay nagbibigay ng madaling paraan para maisagawa ang amplification na ito, ngunit maaaring hindi tumpak at maaaring humantong sa maling mga resulta. Tingnan ang artikulo ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), at partikular ang pahina 4 ng supplementary information para sa mga detalye. Ang probabilistic error amplification ay nagbibigay ng mas tumpak na diskarte sa error amplification sa pamamagitan ng noise learning.

Ang PEA ay isang mas sopistikadong teknik na nagsasagawa ng mga paunang eksperimento upang muling itayo ang noise at pagkatapos ay ginagamit ang impormasyong ito upang maisagawa ang isang tumpak na amplification. Nagsisimula ito sa pag-aaral ng twirled noise model ng bawat layer ng mga entangling gate sa circuit bago sila patakbuhin (tingnan ang [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) para sa mga kaugnay na opsyon ng learning). Pagkatapos ng learning phase, ang mga circuit ay isinasagawa sa bawat noise factor, kung saan ang bawat entangling layer ng mga circuit ay pine-amplify sa pamamagitan ng probabilistic na pag-inject ng single-qubit noise na proporsyonal sa kaukulang natutunan na noise model. Tingnan ang artikulo ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) para sa karagdagang detalye.

Ang PEA ay binubuo ng tatlong yugto:
1. _Learning_: Natututo ang twirled noise model ng bawat layer ng mga entangling gate sa circuit.
1. _Noise amplification_: Ang orihinal na quantum circuit ay isinasagawa nang maraming beses sa iba't ibang noise factor.
2. _Extrapolation_: Ang ideal na resulta ay tinatantya sa pamamagitan ng pag-extrapolate ng mga maingay na resulta ng expectation value patungo sa zero-noise limit.

Para sa mga eksperimento sa utility-scale, ang PEA ay madalas na pinakamainam na pagpipilian.

Dahil ang PEA ay isang ZNE noise amplification technique, kailangan mo ring i-enable ang ZNE sa pamamagitan ng pag-set ng `resilience.zne_mitigation = True`. Ang iba pang mga opsyon ng [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) ay maaari ding gamitin upang i-set ang mga extrapolator, antas ng amplification, at iba pa. Ang PEA ay nangangailangan ng noise model, na awtomatikong nalilikha kapag gumagamit ng mga primitive.

Ang sumusunod na snippet ay nagbibigay ng halimbawa kung saan ginagamit ang PEA upang mapagaan ang resulta ng isang Estimator job:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistic error cancellation (PEC)
Ang Probabilistic error cancellation (PEC) ay isang teknik para mapagaan ang mga error sa pagtatantya ng mga expectation value ng mga observable. Hindi tulad ng ZNE, nagbabalik ito ng walang bias na tantya ng expectation value. Gayunpaman, sa pangkalahatan ito ay nagdudulot ng mas malaking overhead.

Sa PEC, ang epekto ng isang ideal na target circuit ay ipinahayag bilang isang linear combination ng mga maingay na circuit na aktwal na maisasagawa sa practice:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Ang output ng ideal na circuit ay maaaring muling gawin sa pamamagitan ng pagpapatupad ng iba't ibang maingay na circuit instance na kinuha mula sa isang random na ensemble na tinukoy ng linear combination. Kung ang mga coefficient na $\eta_i$ ay bumubuo ng isang probability distribution, maaari itong gamitin nang direkta bilang mga probabilidad ng ensemble. Sa practice, ang ilan sa mga coefficient ay negatibo, kaya bumubuo sila ng isang quasi-probability distribution sa halip. Maaari pa rin itong gamitin upang tukuyin ang isang random na ensemble, ngunit may sampling overhead na kaugnay sa negativity ng quasi-probability distribution, na nailalarawan ng dami

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Ang sampling overhead ay isang multiplicative factor sa bilang ng mga shot na kinakailangan upang matantya ang isang expectation value sa isang partikular na katumpakan, kumpara sa bilang ng mga shot na kakailanganin mula sa ideal na circuit. Nag-e-scale ito nang quadratically kasabay ng $\gamma$, na sa turn ay nag-e-scale nang exponentially kasabay ng depth ng circuit.

Maaaring i-enable ang PEC sa pamamagitan ng pag-set ng `pec_mitigation` sa `True` sa [mga opsyon ng Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para sa Estimator.
Ang mga opsyon ng Qiskit Runtime para sa PEC ay inilalarawan [dito](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Maaaring i-set ang isang limitasyon sa sampling overhead gamit ang opsyon na `max_overhead`. Tandaan na ang paglilimita sa sampling overhead ay maaaring maging sanhi ng katumpakan ng resulta na malampasan ang hiniling na katumpakan. Ang default na halaga ng `max_overhead` ay 100.

Ang sumusunod na code cell ay nagpapakita kung paano i-enable ang PEC at i-set ang opsyon na `max_overhead` para sa estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Mga susunod na hakbang
- Tingnan ang [tutorial](/tutorials/combine-error-mitigation-techniques) sa pagsasama ng mga opsyon ng error mitigation kasama ang Estimator primitive.
- [I-configure ang error mitigation.](configure-error-mitigation)
- [I-configure ang error suppression.](configure-error-suppression)
- Tuklasin ang iba pang [mga opsyon](runtime-options-overview) para sa mga Qiskit Runtime primitive.
- Magpasya kung anong [execution mode](execution-modes) ang patakbuhin ng iyong job.